In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Analyse the differences between methods

## Create union and intersection models

In [ ]:
df_snorkel = pd.read_feather(r"results\snorkel_directly_on_test_mv.feather")
df_rag = pd.read_feather(r"results\test_set_lexical.feather")

print(df_snorkel.columns)
print(df_snorkel.head())

pred_cols = [col for col in df_snorkel.columns if col.startswith("pred_")]
df_snorkel[pred_cols] = df_snorkel[pred_cols].fillna(0).astype(int)
print(df_snorkel.head())

In [ ]:
df_rag.rename(columns={"SC_LLM": "Sc_LLM"}, inplace=True)

In [ ]:
label_map = {
    "C": "CHF",
    "H": "HTN",
    "D": "DM",
    "S": "STROKE",
    "V": "VASCULAR",
    "A": "AGE_65_74",
    "A2": "AGE_75",
    "Sc": "SEX_FEMALE",
}

requested_rag_cols = (
    ["unique_id"]
    + [f"{k}_LLM" for k in label_map.keys()]
    + [f"{k}" for k in label_map.values()]
    + [f"{k}_chunk" for k in label_map.keys()]
)

available_rag_cols = [col for col in requested_rag_cols if col in df_rag.columns]

df_compare = pd.merge(
    df_snorkel, df_rag[available_rag_cols], on="unique_id", how="inner"
)


for short in label_map.keys():
    # Create clean 'snorkel_' and 'rag_' columns within the merged DF
    df_compare[f"snorkel_{short}"] = df_compare[f"pred_{short}"]
    df_compare[f"rag_{short}"] = df_compare[f"{short}_LLM"]

    # Calculate Intersection (Both say Yes) and Union (At least one says Yes)
    df_compare[f"inter_{short}"] = (
        df_compare[f"snorkel_{short}"].astype(int)
        & df_compare[f"rag_{short}"].astype(int)
    ).astype(int)

    df_compare[f"union_{short}"] = (
        df_compare[f"snorkel_{short}"].astype(int)
        | df_compare[f"rag_{short}"].astype(int)
    ).astype(int)

print(f"Merge successful. Rows compared: {len(df_compare)}")
print(f"Sample HTN Match: {df_compare[['snorkel_H', 'rag_H']].head(5)}")

In [ ]:
print(df_compare.columns)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support


def evaluate_prefix(df, prefix):
    short_codes = list(label_map.keys())
    gt_cols = [label_map[s] for s in short_codes]
    pred_cols = [f"{prefix}{s}" for s in short_codes]

    y_true_mat = df[gt_cols].values.astype(int)
    y_pred_mat = df[pred_cols].values.astype(int)

    p, r, f, _ = precision_recall_fscore_support(
        y_true_mat, y_pred_mat, average=None, zero_division=0
    )

    results_df = pd.DataFrame({"precision": p, "recall": r, "f1": f}, index=gt_cols)

    results_df.loc["MACRO_AVG"] = results_df.mean()

    y_true_flat = y_true_mat.ravel()
    y_pred_flat = y_pred_mat.ravel()
    p_mic, r_mic, f_mic, _ = precision_recall_fscore_support(
        y_true_flat, y_pred_flat, average="binary", zero_division=0
    )

    results_df.loc["MICRO_AVG"] = [p_mic, r_mic, f_mic]

    return results_df.round(3).rename_axis("Component").reset_index()

In [ ]:
snorkel_results = evaluate_prefix(df_compare, "snorkel_")
rag_results = evaluate_prefix(df_compare, "rag_")
intersection_results = evaluate_prefix(df_compare, "inter_")
union_results = evaluate_prefix(df_compare, "union_")

snorkel_results["method"] = "Heuristic voting"
rag_results["method"] = "LLM"
intersection_results["method"] = "Intersection"
union_results["method"] = "Union"

df_comparison_performance = pd.concat(
    [snorkel_results, rag_results, intersection_results, union_results]
)

df_comparison_performance

In [ ]:
from sklearn.metrics import cohen_kappa_score


def compute_kappa(df, prefix1, prefix2):
    results = {}

    for short in label_map.keys():
        y1 = df[f"{prefix1}{short}"]
        y2 = df[f"{prefix2}{short}"]

        kappa = cohen_kappa_score(y1, y2)
        kappa = round(kappa, 3)

        results[short] = {"kappa": kappa}

    return pd.DataFrame(results).T


kappa_snorkel_rag = compute_kappa(df_compare, "snorkel_", "rag_")
kappa_snorkel_rag

In [ ]:
def pivot_metric_table(comparison_df, metric="f1"):
    df = comparison_df.copy()

    if "Component" not in df.columns:
        if df.index.name == "Component" or df.index.name is None:
            df = df.rename_axis("Component").reset_index()
        else:
            raise KeyError("Could not find 'Component' in columns or index.")

    wide_df = df.pivot(index="Component", columns="method", values=metric).reset_index()

    wide_df.columns.name = None
    wide_df = wide_df.round(3)

    method_order = ["Snorkel", "RAG", "Union", "Intersection"]
    cols = ["Component"] + [m for m in method_order if m in wide_df.columns]

    return wide_df[cols]

In [ ]:
pivot_metric_table(df_comparison_performance, metric="f1")

In [ ]:
pivot_metric_table(df_comparison_performance, metric="precision")

In [ ]:
pivot_metric_table(df_comparison_performance, metric="recall")

In [ ]:
def plot_metrics(comparison_df, metrics=["precision", "recall", "f1"]):
    component_order = [
        "CHF",
        "HTN",
        "AGE_75",
        "DM",
        "STROKE",
        "VASCULAR",
        "AGE_65_74",
        "SEX_FEMALE",
        "MACRO_AVG",
        "MICRO_AVG",
    ]
    method_order = ["Snorkel", "RAG", "Union", "Intersection"]

    sns.set_style("white")

    for metric in metrics:
        df = comparison_df.copy()

        if "Component" not in df.columns:
            if df.index.name == "Component" or df.index.name is None:
                df = df.rename_axis("Component").reset_index()

        pivot_df = df.pivot(index="Component", columns="method", values=metric)
        pivot_df = pivot_df.reindex(component_order)
        pivot_df = pivot_df[[m for m in method_order if m in pivot_df.columns]]

        plot_df = pivot_df.reset_index().melt(
            id_vars="Component", var_name="Method", value_name=metric
        )

        plt.figure(figsize=(14, 6))

        ax = sns.barplot(
            data=plot_df,
            x="Component",
            y=metric,
            hue="Method",
            hue_order=method_order,
            gap=0.1,
        )

        plt.gca().yaxis.grid(True, linestyle="--", alpha=0.7, zorder=0)
        ax.set_axisbelow(True)

        plt.title(f"{metric.upper()} comparison", fontsize=14, fontweight="bold")
        plt.ylabel(metric)
        plt.xlabel("Component")
        plt.xticks(rotation=45)
        plt.ylim(0, 1.05)

        plt.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()


plot_metrics(df_comparison_performance)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


def plot_dumbbell_final_v3(df, metric="precision"):
    name_map = {
        "CHF": "C",
        "HTN": "H",
        "AGE_75": "A$_{2}$",
        "DM": "D",
        "STROKE": "S$_{2}$",
        "VASCULAR": "V",
        "AGE_65_74": "A",
        "SEX_FEMALE": "Sc",
    }

    method_markers = {"Snorkel": "o", "RAG": "s", "Union": "^", "Intersection": "D"}

    components_list = [
        "CHF",
        "HTN",
        "AGE_75",
        "DM",
        "STROKE",
        "VASCULAR",
        "AGE_65_74",
        "SEX_FEMALE",
    ]
    plot_order = components_list[::-1]
    display_names = [name_map.get(c, c) for c in plot_order]
    method_order = ["Snorkel", "RAG", "Union", "Intersection"]

    plt.figure(figsize=(10, 7))
    sns.set_style("white")

    for i, comp in enumerate(plot_order):
        row_values = df[df["Component"] == comp][metric]
        if not row_values.empty:
            plt.hlines(
                y=i,
                xmin=row_values.min(),
                xmax=row_values.max(),
                color="lightgray",
                linewidth=2,
                zorder=1,
            )

    for method in method_order:
        method_data = df[df["method"] == method]
        relevant_data = method_data[method_data["Component"].isin(plot_order)]
        relevant_data = (
            relevant_data.set_index("Component").reindex(plot_order).reset_index()
        )

        plt.scatter(
            relevant_data[metric],
            range(len(plot_order)),
            label=method,
            s=130,
            marker=method_markers[method],
            edgecolor="white",
            linewidth=1.2,
            alpha=0.8,
            zorder=5,
        )

    plt.yticks(range(len(plot_order)), display_names)
    plt.title(
        f"Performance Variance Across CHA$_{2}$DS$_{2}$-VASc Components by Extraction Method: {metric.upper()}",
        loc="left",
        fontsize=14,
        fontweight="bold",
        pad=25,
    )
    plt.xlim(0.7, 1.01)
    plt.xlabel(f"{metric.capitalize()} Score")

    sns.despine(left=True, bottom=False)
    plt.grid(axis="x", linestyle="--", alpha=0.3)

    plt.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)
    plt.show()


plot_dumbbell_final_v3(df_comparison_performance, metric="f1")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


def generate_thesis_composite_figure(df):
    """
    Creates a single, professional 3-panel figure comparing Precision,
    Recall, and F1-Score in a dumbbell/dot plot format.

    Parameters:
    -----------
    df : pandas.DataFrame
        The long-format DataFrame containing performance metrics.
        Required columns: 'Component' (e.g., 'CHF', 'MACRO_AVG'),
        'method' (e.g., 'Snorkel'), 'precision', 'recall', 'f1'.
    """

    name_map = {
        "CHF": "C",
        "HTN": "H",
        "AGE_75": "A$_{2}$",
        "DM": "D",
        "STROKE": "S$_{2}$",
        "VASCULAR": "V",
        "AGE_65_74": "A",
        "SEX_FEMALE": "Sc",
    }

    # Shapes are crucial for grayscale compatibility
    method_markers = {
        "Heuristic voting": "o",
        "LLM": "s",
        "Union": "^",
        "Intersection": "D",
    }

    components_list = [
        "CHF",
        "HTN",
        "AGE_75",
        "DM",
        "STROKE",
        "VASCULAR",
        "AGE_65_74",
        "SEX_FEMALE",
    ]
    plot_order = components_list[::-1]
    display_names = [name_map.get(c, c) for c in plot_order]

    method_order = ["Heuristic voting", "LLM", "Union", "Intersection"]

    metrics_to_plot = ["precision", "recall", "f1"]
    titles = ["Precision", "Recall", "F1-Score"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    sns.set_style("white")

    fig.suptitle(
        "Comparative Evaluation of Automated CHA$_{2}$DS$_{2}$-VASc Scoring Logic",
        fontsize=18,
        fontweight="bold",
        y=0.98,
    )

    for ax, metric, title_text in zip(axes, metrics_to_plot, titles):

        for i, comp in enumerate(plot_order):
            row_values = df[df["Component"] == comp][metric]
            if not row_values.empty:
                ax.hlines(
                    y=i,
                    xmin=row_values.min(),
                    xmax=row_values.max(),
                    color="lightgray",
                    linewidth=2.5,
                    zorder=1,
                )

        for method in method_order:
            method_data = df[df["method"] == method]
            relevant_data = method_data[method_data["Component"].isin(plot_order)]
            relevant_data = (
                relevant_data.set_index("Component").reindex(plot_order).reset_index()
            )

            ax.scatter(
                relevant_data[metric],
                range(len(plot_order)),
                label=method,
                s=150,
                marker=method_markers[method],
                edgecolor="white",
                linewidth=1.2,
                alpha=1,  # Crucial for visible overlap at 1.0
                zorder=5,
            )

        if ax == axes[0]:
            ax.set_yticks(range(len(plot_order)))
            ax.set_yticklabels(display_names, fontsize=12)
        else:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)

        ax.set_title(title_text, fontsize=15, fontweight="semibold")
        ax.set_xlim(0.70, 1.02)  # Fixed scale for consistent visual comparison
        ax.set_xlabel(f"{title_text}", fontsize=13)

        sns.despine(ax=ax, left=True, bottom=False)
        ax.grid(axis="x", linestyle="--", alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        title="Extraction Method",
        bbox_to_anchor=(0.5, 0.02),
        loc="upper center",
        ncol=4,
        frameon=True,
        fontsize=12,
        title_fontsize=13,
    )

    plt.tight_layout()
    plt.subplots_adjust(top=0.88, bottom=0.15)

    output_filename = "cha2ds2vasc_results_grid.png"
    plt.savefig(output_filename, dpi=300, bbox_inches="tight")
    print(f"Thesis composite figure successfully saved as: {output_filename}")
    plt.show()


try:
    generate_thesis_composite_figure(df_comparison_performance)
except NameError:
    print(
        "Error: DataFrame 'df_comparison_performance' not found. Please ensure your data preparation code runs before calling this function."
    )
except Exception as e:
    print(f"An unexpected error occurred during plotting: {e}")

## Inspect cases where Snorkel and RAG disagree


In [ ]:
results_analysis = []

for short, full_name in label_map.items():
    s_col = f"snorkel_{short}"
    r_col = f"rag_{short}"
    gt_col = full_name

    # --- 1. Mismatch Analysis (One is right, one is wrong) ---
    mismatch_mask = df_compare[s_col] != df_compare[r_col]
    df_mismatch = df_compare[mismatch_mask]
    total_mismatch = len(df_mismatch)

    s_wins = (
        (df_mismatch[s_col] == df_mismatch[gt_col]).sum() if total_mismatch > 0 else 0
    )
    r_wins = (
        (df_mismatch[r_col] == df_mismatch[gt_col]).sum() if total_mismatch > 0 else 0
    )

    # --- 2. Consensus Error Analysis (Both agreed, but both are wrong) ---
    # This is the 'Hard' category: systemic failure or Ground Truth ambiguity
    both_wrong_mask = (df_compare[s_col] == df_compare[r_col]) & (
        df_compare[s_col] != df_compare[gt_col]
    )
    df_both_wrong = df_compare[both_wrong_mask]
    total_both_wrong = len(df_both_wrong)

    results_analysis.append(
        {
            "Component": full_name,
            "Total Mismatches": total_mismatch,
            "Heuristic voting is right": s_wins,
            "LLM is right": r_wins,
            "Both Wrong": total_both_wrong,
            "Winner (Mismatches)": (
                "LLM"
                if r_wins > s_wins
                else ("Heuristic voting" if s_wins > r_wins else "Tie")
            ),
        }
    )

analysis_df = pd.DataFrame(results_analysis)
# Sorting by systemic errors can reveal which components are 'hardest' for AI
print(analysis_df.sort_values("Both Wrong", ascending=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch


def plot_conflict_analysis_consistent(analysis_df):
    # 1. Map with LaTeX for consistency
    rename_dict = {
        "CHF": "C",
        "HTN": "H",
        "AGE_75": "A$_{2}$",
        "DM": "D",
        "STROKE": "S$_{2}$",
        "VASCULAR": "V",
        "AGE_65_74": "A",
        "SEX_FEMALE": "Sc",
    }

    # 2. Data Preparation
    cha_order = [
        "CHF",
        "HTN",
        "AGE_75",
        "DM",
        "STROKE",
        "VASCULAR",
        "AGE_65_74",
        "SEX_FEMALE",
    ]
    plot_order = [c for c in cha_order if c in analysis_df["Component"].values][::-1]

    plot_df = analysis_df.set_index("Component").reindex(plot_order).fillna(0)
    plot_df.index = [rename_dict.get(idx, idx) for idx in plot_df.index]

    # These categories represent the "Winner" of the disagreement
    categories = ["Heuristic voting is right", "LLM is right", "Both Wrong"]

    # 3. CONSISTENT COLORS FROM DUMBBELL PLOT
    # Blue for Snorkel, Orange for RAG (LLM), Gray for failure cases
    colors = ["#1f77b4", "#ff7f0e", "#d3d3d3"]

    # 4. Plot Setup
    sns.set_theme(style="white")
    fig, ax = plt.subplots(figsize=(10, 6))

    plot_df[categories].plot(
        kind="barh",
        stacked=True,
        ax=ax,
        color=colors,
        width=0.75,
        edgecolor="white",
        linewidth=1,
        legend=False,
    )

    # 5. Label Logic
    yticks = ax.get_yticks()
    total_max = plot_df[categories].sum(axis=1).max()

    for i, (idx, row) in enumerate(plot_df.iterrows()):
        current_y = yticks[i]
        left_offset = 0
        for col_idx, col_name in enumerate(categories):
            val = row[col_name]

            if val > 0:
                center_x = left_offset + (val / 2)

                # Snorkel and RAG get white text, Gray gets dark text
                t_color = "white" if col_idx in [0, 1] else "#333333"
                f_size = 9 if (val / total_max) < 0.04 else 11

                ax.text(
                    center_x,
                    current_y,
                    f"{int(val)}",
                    va="center",
                    ha="center",
                    color=t_color,
                    fontweight="bold",
                    fontsize=f_size,
                )

            left_offset += val

    # 6. Aesthetics
    sns.despine(left=True, bottom=False)
    ax.grid(axis="x", linestyle="--", alpha=0.3, color="gray")

    ax.set_xlabel("Number of Disagreement Cases (n)", fontsize=12, fontweight="bold")
    ax.set_title(
        "Resolution of Model Disagreements by Component (test set)",
        fontsize=14,
        pad=20,
        loc="left",
        fontweight="bold",
    )

    ax.tick_params(axis="y", labelsize=14, length=0)
    ax.tick_params(axis="x", labelsize=11)

    # 7. Legend
    legend_labels = ["Heuristic voting is right", "LLM is right", "Both Wrong"]
    legend_elements = [
        Patch(facecolor=c, label=l) for c, l in zip(colors, legend_labels)
    ]

    leg = ax.legend(
        handles=legend_elements,
        title="Correct Model (Matched with GT)",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=True,
    )

    plt.tight_layout()
    plt.savefig("conflict_analysis_consistent.png", dpi=300, bbox_inches="tight")
    plt.show()


# Run
plot_conflict_analysis_consistent(analysis_df)

In [ ]:
disagreement_results = []

for short, full_name in label_map.items():
    s_col = f"snorkel_{short}"
    r_col = f"rag_{short}"

    # Calculate where they differ
    mismatch_count = (df_compare[s_col] != df_compare[r_col]).sum()
    mismatch_pct = (df_compare[s_col] != df_compare[r_col]).mean() * 100

    disagreement_results.append(
        {
            "Component": full_name,
            "Mismatches": mismatch_count,
            "Disagreement %": round(mismatch_pct, 2),
        }
    )

disagreement_df = pd.DataFrame(disagreement_results)
print("===== SNORKEL VS RAG DISAGREEMENT =====")
print(disagreement_df.sort_values("Disagreement %", ascending=False))

In [ ]:
icd_map = {
    "H": [
        "diagnose_i10",
        "diagnose_i11",
        "diagnose_i12",
        "diagnose_i13",
        "diagnose_i15",
    ],
    "C": [
        "diagnose_i501",
        "diagnose_i502",
        "diagnose_i503",
        "diagnose_i504",
        "diagnose_i508",
        "diagnose_i509",
    ],
    "S": ["diagnose_i69", "diagnose_z8673"],
    "D": [
        "diagnose_e10",
        "diagnose_e11",
        "diagnose_e12",
        "diagnose_e13",
        "diagnose_e14",
    ],
}

In [ ]:
import textwrap


def inspect_mismatches(df, short_code, num_samples=5):
    # 1. Column definitions
    s_col = f"snorkel_{short_code}"
    r_col = f"rag_{short_code}"
    fired_col = f"fired_lfs_{short_code}"
    chunk_col = f"{short_code}_chunk"

    # 2. Filter for disagreements
    mismatch_idx = df[s_col] != df[r_col]
    mismatches = df[mismatch_idx].copy()

    # Setup for wrapping
    wrapper = textwrap.TextWrapper(
        width=100, initial_indent="      ", subsequent_indent="      "
    )

    header = (
        f" QUALITATIVE ERROR ANALYSIS: {label_map.get(short_code, short_code).upper()} "
    )
    print("\n" + "=" * 85)
    print(header.center(85, "="))
    print(f" Total Mismatches: {len(mismatches)}".center(85, "="))
    print("=" * 85 + "\n")

    if len(mismatches) == 0:
        print("No mismatches found for this component.")
        return

    # 3. Sampling
    samples = mismatches.sample(min(num_samples, len(mismatches)))
    relevant_icds = icd_map.get(short_code, [])

    for i, row in samples.iterrows():
        print(f"▶ Patient ID: {row.get('unique_id', i)}")

        # --- PREDICTION SUMMARY ---
        gt_val = row.get(label_map[short_code], "N/A")
        print(
            f"  [LABELS]    Snorkel: {int(row[s_col])} | RAG-LLM: {int(row[r_col])} | Ground truth: {gt_val}"
        )

        # --- SNORKEL LOGIC ---
        if fired_col in df.columns:
            lfs = row[fired_col]
            lf_display = (
                ", ".join(map(str, lfs))
                if isinstance(lfs, (list, np.ndarray)) and len(lfs) > 0
                else "NONE"
            )
            print(f"  [HEURISTICS]   Fired LFs: {lf_display}")

        # --- RAG EVIDENCE (Pretty Printed) ---
        print(f"  [LLM EVIDENCE]:")
        if chunk_col in df.columns and pd.notnull(row[chunk_col]):
            evidence = str(row[chunk_col]).strip()
            # Wrap the evidence chunk
            print(wrapper.fill(f'"{evidence}"'))
        else:
            print("      No specific evidence chunk recorded.")

        # --- ICD STATUS ---
        if relevant_icds:
            found_icds = [code for code in relevant_icds if row.get(code) == 1]
            status = f"POSITIVE ({', '.join(found_icds)})" if found_icds else "NEGATIVE"
            print(f"  [ICD CODES]: {status}")

        # --- FULL TEXT PREVIEW (Pretty Printed) ---
        print(f"  [TEXT PREVIEW]:")
        text_content = str(row["text"]) if "text" in df.columns else "N/A"
        # Clean text: remove tabs and extra spaces
        clean_text = " ".join(text_content[0:2000].split())
        print(wrapper.fill(f"{clean_text}..."))

        print("\n" + "·" * 85 + "\n")


# Run it
inspect_mismatches(df_compare, "V", num_samples=10)

# Create intersection fusion on the training set

In [ ]:
import pandas as pd

label_map = {
    "C": "CHF",
    "H": "HTN",
    "D": "DM",
    "S": "STROKE",
    "V": "VASCULAR",
    "A": "AGE_65_74",
    "A2": "AGE_75",
    "Sc": "SEX_FEMALE",
}


df_snorkel = pd.read_feather(r"results\snorkel_directly_on_train_mv.feather")
df_rag = pd.read_feather(r"results\train_set_lexical_v2.feather")

df_rag.rename(columns={"SC_LLM": "Sc_LLM"}, inplace=True)

snorkel_cols = ["unique_id"] + [f"pred_{k}" for k in label_map.keys()]
df_compare = pd.merge(df_rag, df_snorkel[snorkel_cols], on="unique_id", how="inner")

print(f"Final Merged Shape: {df_compare.shape}")

for short in label_map.keys():
    df_compare[f"snorkel_{short}"] = df_compare[f"pred_{short}"]
    df_compare[f"rag_{short}"] = df_compare[f"{short}_LLM"]

    s_bool = df_compare[f"snorkel_{short}"].astype(bool)
    r_bool = df_compare[f"rag_{short}"].astype(bool)

    df_compare[f"inter_{short}"] = (s_bool & r_bool).astype(int)
    df_compare[f"union_{short}"] = (s_bool | r_bool).astype(int)

print(f"Merged Shape: {df_compare.shape}")
print(
    f"Agreement on CHF (C): {(df_compare['snorkel_C'] == df_compare['rag_C']).mean():.2%}"
)

In [ ]:
# 1. Extract the single row
row = df_compare.loc[df_compare["unique_id"] == 208].iloc[0]

print(f"{'='*30} ROW DETAILS: ID 300 {'='*30}")
for col_name, value in row.items():
    print(f"{col_name:30}: {value}")
print("=" * 75)

In [ ]:
weights = {
    "C": 1,
    "H": 1,
    "A2": 2,  # Age >= 75
    "D": 1,
    "S": 2,  # Prior Stroke / TIA
    "V": 1,
    "A": 1,  # Age 65-74
    "Sc": 1,
}

# Apply weights and then sum
df_compare["reconstructed_score_intersection"] = sum(
    df_compare[f"inter_{short}"] * weight for short, weight in weights.items()
)

print(df_compare[["reconstructed_score_intersection"]].head())

In [ ]:
df_compare.to_feather("results/intersection_model_train.feather")